In [1]:
from pathlib import Path
from urllib.request import urlretrieve

import polars as pl

PROJECT_ROOT = (
    Path.cwd().parent
    if Path.cwd().name == "notebooks"
    else Path.cwd()
)

DATA_DIR = PROJECT_ROOT / "data"
METADATA_PATH = DATA_DIR / "isd-history.csv"

METADATA_URL = (
    "https://www.ncei.noaa.gov/pub/data/noaa/isd-history.csv"
)

DATA_DIR.mkdir(parents=True, exist_ok=True)

if not METADATA_PATH.exists():
    urlretrieve(METADATA_URL, METADATA_PATH)

stations = pl.read_csv(
    METADATA_PATH,
    schema_overrides={
        "USAF": pl.String,
        "WBAN": pl.String,
    },
)

stations.shape

(29661, 11)

In [2]:
indonesia_stations = (
    stations
    .filter(pl.col("CTRY") == "ID")
    .filter(
        (pl.col("END") >= 20150101)
        & (pl.col("BEGIN") <= 20241231)
    )
    .select(
        "USAF",
        "WBAN",
        "STATION NAME",
        "ICAO",
        "LAT",
        "LON",
        "ELEV(M)",
        "BEGIN",
        "END",
    )
    .sort("STATION NAME")
)

print(f"Jumlah stasiun Indonesia: {indonesia_stations.height}")
indonesia_stations.head(10)

Jumlah stasiun Indonesia: 117


USAF,WBAN,STATION NAME,ICAO,LAT,LON,ELEV(M),BEGIN,END
str,str,str,str,str,str,str,i64,i64
"""966336""","""99999""","""ADISUMARMO INTL / ADI SOEMARMO""","""WAHQ""","""-07.500""","""+110.750""","""+0127.4""",20050111,20210319
"""973200""","""99999""","""ALOR/MALI/KALABAHI""","""WRKM""","""-08.217""","""+124.567""","""+0012.0""",19460101,20250307
"""977220""","""99999""","""AMAHAI""","""WAPA""","""-03.350""","""+128.883""","""+0010.0""",19801001,20250307
"""974300""","""99999""","""BABULLAH""","""WAMT""","""+00.831""","""+127.381""","""+0014.9""",19510101,20250824
"""971925""","""99999""","""BALI INTL / I GUSTI NGURAH RAI""","""WADD""","""-08.748""","""+115.167""","""+0004.3""",20050104,20250824
"""969870""","""99999""","""BANYUWANGI""","""""","""-08.217""","""+114.383""","""+0005.0""",19560101,20250307
"""960870""","""99999""","""BATAM/HANG NADIM""","""""","""+01.117""","""+104.117""","""+0024.0""",19860103,20250307
"""971920""","""99999""","""BAU-BAU/BETO AMBIRI""","""WAAB""","""-05.467""","""+122.617""","""+0002.0""",19800501,20250824
"""960750""","""99999""","""BINAKA""","""WIMB""","""+01.166""","""+097.705""","""+0006.1""",19790601,20250307


In [3]:
from concurrent.futures import ThreadPoolExecutor, as_completed
from urllib.error import HTTPError, URLError
from urllib.request import urlretrieve

RAW_DIR = DATA_DIR / "raw" / "noaa_global_hourly"
RAW_DIR.mkdir(parents=True, exist_ok=True)

YEAR = 2024
BASE_URL = f"https://www.ncei.noaa.gov/data/global-hourly/access/{YEAR}"

stations_2024 = indonesia_stations.filter(
    (pl.col("BEGIN") <= 20241231)
    & (pl.col("END") >= 20240101)
)

download_jobs = []

for row in stations_2024.iter_rows(named=True):
    station_id = f"{row['USAF']}{row['WBAN']}"
    filename = f"{station_id}.csv"

    download_jobs.append(
        {
            "url": f"{BASE_URL}/{filename}",
            "path": RAW_DIR / filename,
        }
    )

print(f"File yang akan diperiksa: {len(download_jobs)}")

File yang akan diperiksa: 114


In [4]:
def download_file(job):
    if job["path"].exists():
        return "sudah_ada", job["path"].name

    try:
        urlretrieve(job["url"], job["path"])
        return "berhasil", job["path"].name
    except (HTTPError, URLError) as error:
        return "gagal", f"{job['path'].name}: {error}"


results = []

with ThreadPoolExecutor(max_workers=6) as executor:
    futures = [
        executor.submit(download_file, job)
        for job in download_jobs
    ]

    for number, future in enumerate(as_completed(futures), start=1):
        status, detail = future.result()
        results.append((status, detail))

        if number % 10 == 0 or number == len(futures):
            print(f"Progres: {number}/{len(futures)}")

successful = sum(
    status in {"berhasil", "sudah_ada"}
    for status, _ in results
)
failed = [
    detail
    for status, detail in results
    if status == "gagal"
]

print(f"Berhasil/tersedia: {successful}")
print(f"Gagal: {len(failed)}")

if failed:
    print("Contoh kegagalan:", failed[:5])

Progres: 10/114
Progres: 20/114
Progres: 30/114
Progres: 40/114
Progres: 50/114
Progres: 60/114
Progres: 70/114
Progres: 80/114
Progres: 90/114
Progres: 100/114
Progres: 110/114
Progres: 114/114
Berhasil/tersedia: 114
Gagal: 0


In [5]:
csv_files = sorted(RAW_DIR.glob("*.csv"))

total_size_bytes = sum(
    file.stat().st_size
    for file in csv_files
)

total_rows = 0

for file in csv_files:
    with file.open("rb") as csv_file:
        row_count = sum(1 for _ in csv_file) - 1
        total_rows += max(row_count, 0)

print(f"Jumlah file: {len(csv_files)}")
print(f"Ukuran total: {total_size_bytes / (1024**2):,.2f} MB")
print(f"Jumlah baris: {total_rows:,}")

Jumlah file: 114
Ukuran total: 275.42 MB
Jumlah baris: 668,752


In [6]:
RAW_ROOT = DATA_DIR / "raw" / "noaa_global_hourly"
YEAR_2024_DIR = RAW_ROOT / "2024"

YEAR_2024_DIR.mkdir(parents=True, exist_ok=True)

moved = 0

for file in RAW_ROOT.glob("*.csv"):
    destination = YEAR_2024_DIR / file.name

    if not destination.exists():
        file.replace(destination)
        moved += 1

print(f"File 2024 yang dipindahkan: {moved}")

File 2024 yang dipindahkan: 114


In [7]:
YEAR = 2023
YEAR_DIR = RAW_ROOT / str(YEAR)
YEAR_DIR.mkdir(parents=True, exist_ok=True)

BASE_URL = f"https://www.ncei.noaa.gov/data/global-hourly/access/{YEAR}"

stations_2023 = indonesia_stations.filter(
    (pl.col("BEGIN") <= 20231231)
    & (pl.col("END") >= 20230101)
)

download_jobs_2023 = []

for row in stations_2023.iter_rows(named=True):
    station_id = f"{row['USAF']}{row['WBAN']}"
    filename = f"{station_id}.csv"

    download_jobs_2023.append(
        {
            "url": f"{BASE_URL}/{filename}",
            "path": YEAR_DIR / filename,
        }
    )

print(f"File 2023 yang akan diperiksa: {len(download_jobs_2023)}")

File 2023 yang akan diperiksa: 114


In [15]:
results_2023 = []

with ThreadPoolExecutor(max_workers=6) as executor:
    futures = [
        executor.submit(download_file, job)
        for job in download_jobs_2023
    ]

    for number, future in enumerate(as_completed(futures), start=1):
        status, detail = future.result()
        results_2023.append((status, detail))

        print(
            f"{number}/{len(futures)} - "
            f"{status}: {detail}"
        )

successful_2023 = sum(
    status in {"berhasil", "sudah_ada"}
    for status, _ in results_2023
)

failed_2023 = [
    detail
    for status, detail in results_2023
    if status == "gagal"
]

print(f"\nBerhasil/tersedia: {successful_2023}")
print(f"Gagal: {len(failed_2023)}")

if failed_2023:
    print("Contoh kegagalan:", failed_2023[:5])

1/114 - sudah_ada: 97192599999.csv
2/114 - sudah_ada: 97320099999.csv
3/114 - sudah_ada: 96987099999.csv
4/114 - sudah_ada: 97722099999.csv
5/114 - sudah_ada: 97430099999.csv
6/114 - sudah_ada: 96087099999.csv
7/114 - sudah_ada: 97192099999.csv
8/114 - sudah_ada: 96075099999.csv
9/114 - sudah_ada: 97016099999.csv
10/114 - sudah_ada: 96751099999.csv
11/114 - sudah_ada: 97086099999.csv
12/114 - sudah_ada: 96805099999.csv
13/114 - sudah_ada: 96791099999.csv
14/114 - sudah_ada: 96179099999.csv
15/114 - sudah_ada: 97230099999.csv
16/114 - sudah_ada: 97372099999.csv
17/114 - sudah_ada: 97780099999.csv
18/114 - sudah_ada: 97372599999.csv
19/114 - sudah_ada: 97630099999.csv
20/114 - sudah_ada: 96253099999.csv
21/114 - sudah_ada: 97748099999.csv
22/114 - sudah_ada: 97560099999.csv
23/114 - sudah_ada: 97406099999.csv
24/114 - sudah_ada: 96249099999.csv
25/114 - sudah_ada: 96781099999.csv
26/114 - sudah_ada: 96749599999.csv
27/114 - sudah_ada: 96645099999.csv
28/114 - sudah_ada: 96745099999.csv
2

In [16]:
csv_files = sorted(RAW_ROOT.rglob("*.csv"))

total_size_bytes = sum(
    file.stat().st_size
    for file in csv_files
)

total_rows = 0
rows_by_year = {}

for file in csv_files:
    with file.open("rb") as csv_file:
        row_count = max(sum(1 for _ in csv_file) - 1, 0)

    total_rows += row_count
    rows_by_year[file.parent.name] = (
        rows_by_year.get(file.parent.name, 0) + row_count
    )

print(f"Jumlah file: {len(csv_files)}")
print(f"Ukuran total: {total_size_bytes / (1024**2):,.2f} MB")
print(f"Jumlah baris: {total_rows:,}")
print(f"Baris per tahun: {rows_by_year}")

Jumlah file: 228
Ukuran total: 523.96 MB
Jumlah baris: 1,273,803
Baris per tahun: {'2023': 605051, '2024': 668752}


In [14]:
print("Kernel berjalan")

Kernel berjalan


# Profiling Dataset Cuaca Indonesia 2023–2024

Notebook ini melakukan profiling awal terhadap data mentah NOAA Global
Hourly dari stasiun meteorologi Indonesia. Data mentah tetap dipertahankan
dan belum dibersihkan pada tahap ini.


In [17]:
from pathlib import Path

import polars as pl

PROJECT_ROOT = (
    Path.cwd().parent
    if Path.cwd().name == "notebooks"
    else Path.cwd()
)

RAW_ROOT = (
    PROJECT_ROOT
    / "data"
    / "raw"
    / "noaa_global_hourly"
)

csv_files = sorted(RAW_ROOT.rglob("*.csv"))

CORE_COLUMNS = [
    "STATION",
    "DATE",
    "LATITUDE",
    "LONGITUDE",
    "ELEVATION",
    "NAME",
    "REPORT_TYPE",
    "QUALITY_CONTROL",
    "WND",
    "CIG",
    "VIS",
    "TMP",
    "DEW",
    "SLP",
]

schema_overrides = {
    column: pl.String
    for column in CORE_COLUMNS
}

lazy_frames = [
    pl.scan_csv(
        file,
        schema_overrides=schema_overrides,
        ignore_errors=True,
    ).select(CORE_COLUMNS)
    for file in csv_files
]

weather_lazy = pl.concat(
    lazy_frames,
    how="vertical_relaxed",
)

print(f"Jumlah file terdeteksi: {len(csv_files)}")
print("Lazy pipeline berhasil dibuat.")

Jumlah file terdeteksi: 228
Lazy pipeline berhasil dibuat.


In [18]:
print("Skema dataset:")
print(weather_lazy.collect_schema())

weather_lazy.head(5).collect()

Skema dataset:
Schema({'STATION': String, 'DATE': String, 'LATITUDE': String, 'LONGITUDE': String, 'ELEVATION': String, 'NAME': String, 'REPORT_TYPE': String, 'QUALITY_CONTROL': String, 'WND': String, 'CIG': String, 'VIS': String, 'TMP': String, 'DEW': String, 'SLP': String})


STATION,DATE,LATITUDE,LONGITUDE,ELEVATION,NAME,REPORT_TYPE,QUALITY_CONTROL,WND,CIG,VIS,TMP,DEW,SLP
str,str,str,str,str,str,str,str,str,str,str,str,str,str
"""96001099999""","""2023-01-01T00:00:00""","""5.874131""","""95.339672""","""113.0""","""MAIMUN SALEH, ID""","""FM-12""","""V020""","""120,1,N,0036,1""","""99999,9,9,N""","""006000,1,9,9""","""+0247,1""","""+0222,1""","""10130,1"""
"""96001099999""","""2023-01-01T03:00:00""","""5.874131""","""95.339672""","""113.0""","""MAIMUN SALEH, ID""","""FM-12""","""V020""","""080,1,N,0036,1""","""99999,9,9,N""","""009000,1,9,9""","""+0262,1""","""+0217,1""","""10147,1"""
"""96001099999""","""2023-01-01T06:00:00""","""5.874131""","""95.339672""","""113.0""","""MAIMUN SALEH, ID""","""FM-12""","""V020""","""100,1,N,0051,1""","""99999,9,9,N""","""009000,1,9,9""","""+0289,1""","""+0230,1""","""10126,1"""
"""96001099999""","""2023-01-01T09:00:00""","""5.874131""","""95.339672""","""113.0""","""MAIMUN SALEH, ID""","""FM-12""","""V020""","""100,1,N,0046,1""","""99999,9,9,N""","""007000,1,9,9""","""+0288,1""","""+0228,1""","""10104,1"""
"""96001099999""","""2023-01-01T12:00:00""","""5.874131""","""95.339672""","""113.0""","""MAIMUN SALEH, ID""","""FM-12""","""V020""","""090,1,N,0026,1""","""99999,9,9,N""","""007000,1,9,9""","""+0262,1""","""+0223,1""","""10119,1"""


In [19]:
summary = (
    weather_lazy
    .select(
        pl.len().alias("jumlah_baris"),
        pl.col("STATION").n_unique().alias("jumlah_stasiun"),
        pl.col("NAME").n_unique().alias("jumlah_nama_stasiun"),
        (
            pl.col("DATE")
            .str.to_datetime(strict=False)
            .min()
            .alias("waktu_awal")
        ),
        (
            pl.col("DATE")
            .str.to_datetime(strict=False)
            .max()
            .alias("waktu_akhir")
        ),
    )
    .collect(engine="streaming")
)

summary

jumlah_baris,jumlah_stasiun,jumlah_nama_stasiun,waktu_awal,waktu_akhir
u32,u32,u32,datetime[μs],datetime[μs]
1273803,114,113,2023-01-01 00:00:00,2024-12-31 23:30:00


null_result = (
    weather_lazy
    .select(
        [
            pl.col(column)
            .null_count()
            .alias(column)
            for column in CORE_COLUMNS
        ]
    )
    .collect(engine="streaming")
)

null_dictionary = null_result.row(0, named=True)

null_profile = (
    pl.DataFrame(
        {
            "kolom": list(null_dictionary.keys()),
            "jumlah_null": list(null_dictionary.values()),
        }
    )
    .with_columns(
        (
            pl.col("jumlah_null")
            / total_rows
            * 100
        )
        .round(2)
        .alias("persentase_null")
    )
    .sort("jumlah_null", descending=True)
)

null_profile

In [20]:
null_result = (
    weather_lazy
    .select(
        [
            pl.col(column)
            .null_count()
            .alias(column)
            for column in CORE_COLUMNS
        ]
    )
    .collect(engine="streaming")
)

null_dictionary = null_result.row(0, named=True)

null_profile = (
    pl.DataFrame(
        {
            "kolom": list(null_dictionary.keys()),
            "jumlah_null": list(null_dictionary.values()),
        }
    )
    .with_columns(
        (
            pl.col("jumlah_null")
            / total_rows
            * 100
        )
        .round(2)
        .alias("persentase_null")
    )
    .sort("jumlah_null", descending=True)
)

null_profile

kolom,jumlah_null,persentase_null
str,i64,f64
"""STATION""",0,0.0
"""DATE""",0,0.0
"""LATITUDE""",0,0.0
"""LONGITUDE""",0,0.0
"""ELEVATION""",0,0.0
…,…,…
"""CIG""",0,0.0
"""VIS""",0,0.0
"""TMP""",0,0.0


In [21]:
observations_by_year = (
    weather_lazy
    .with_columns(
        pl.col("DATE")
        .str.to_datetime(strict=False)
        .alias("datetime")
    )
    .group_by(
        pl.col("datetime")
        .dt.year()
        .alias("tahun")
    )
    .agg(
        pl.len().alias("jumlah_baris"),
        pl.col("STATION")
        .n_unique()
        .alias("jumlah_stasiun"),
    )
    .sort("tahun")
    .collect(engine="streaming")
)

observations_by_year

tahun,jumlah_baris,jumlah_stasiun
i32,u32,u32
2023,605051,114
2024,668752,114


In [22]:
duplicate_result = (
    weather_lazy
    .select(
        pl.struct(CORE_COLUMNS)
        .is_duplicated()
        .sum()
        .alias("jumlah_duplikat_persis")
    )
    .collect(engine="streaming")
)

duplicate_result

jumlah_duplikat_persis
u32
0


In [23]:
report_type_profile = (
    weather_lazy
    .group_by("REPORT_TYPE")
    .agg(
        pl.len().alias("jumlah_baris")
    )
    .sort(
        "jumlah_baris",
        descending=True,
    )
    .collect(engine="streaming")
)

report_type_profile

REPORT_TYPE,jumlah_baris
str,u32
"""FM-12""",643004
"""FM-15""",630775
"""FM-16""",24


## Temuan Profiling Awal

Berdasarkan profiling awal menggunakan Polars, diperoleh temuan berikut:

1. Dataset terdiri dari **1.273.803 baris** pengamatan cuaca yang
   berasal dari **228 file CSV** dengan ukuran total **523,96 MB**.
2. Dataset mencakup **114 ID stasiun** dan **113 nama stasiun unik**.
   Perbedaan jumlah tersebut menunjukkan bahwa terdapat nama stasiun
   yang digunakan oleh lebih dari satu ID stasiun dan perlu diperiksa
   pada tahap cleaning.
3. Rentang waktu pengamatan dimulai pada **1 Januari 2023 pukul 00.00**
   dan berakhir pada **31 Desember 2024 pukul 23.30**.
4. Tahun 2023 memiliki **605.051 baris**, sedangkan tahun 2024 memiliki
   **668.752 baris**. Kedua tahun mencakup 114 stasiun.
5. Tidak ditemukan nilai null yang direpresentasikan sebagai null
   bawaan pada 14 kolom inti.
6. Tidak ditemukan baris yang identik sepenuhnya pada kolom inti.
7. Jenis laporan didominasi oleh `FM-12` sebanyak **643.004 baris** dan
   `FM-15` sebanyak **630.775 baris**. Jenis `FM-16` hanya terdapat pada
   **24 baris**.
8. Beberapa variabel NOAA menggunakan nilai sentinel seperti `+9999`
   atau `99999` untuk menandai data yang tidak tersedia. Oleh karena
   itu, hasil null sebesar nol belum membuktikan bahwa seluruh nilai
   pengamatan lengkap.

## Rencana Pemeriksaan Milestone 2

Tahap cleaning berikutnya perlu mencakup:

- pemisahan nilai pengamatan dan kode kualitas NOAA;
- identifikasi nilai sentinel yang merepresentasikan data hilang;
- konversi kolom waktu dan variabel cuaca ke tipe data yang sesuai;
- pemeriksaan hubungan antara ID dan nama stasiun;
- validasi duplikasi berdasarkan aturan bisnis data pengamatan;
- pemeriksaan nilai ekstrem dan nilai di luar rentang yang masuk akal;
- penyimpanan hasil bersih dalam format Parquet;
- profiling hasil menggunakan DuckDB `SUMMARIZE`.